# Lesson 03 - Exploring LangChain Chains

Chains let us connect multiple LangChain steps together. A chain can start with a prompt template, send the prompt to a model, parse the model output, and then pass that output into another step.

In this lesson, we will learn five core ideas:

- **Basic chain** - connects a prompt, model, and output parser
- **Runnable** - a LangChain object that can receive input and produce output
- **Sequential chain** - passes output from one step into the next step
- **Parallel chain** - runs multiple branches from the same input
- **Conditional chain** - chooses which branch to run based on a condition


## Setup

Before running these examples, make sure your virtual environment is active and your `.env` file contains your OpenAI API key.


In [ ]:
# Install the main packages used in this folder
!pip install langchain-openai python-dotenv

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableSequence, RunnableParallel, RunnableBranch

load_dotenv()

llm = ChatOpenAI(model="gpt-5-nano")

## Understanding The Chain Flow

Most examples in this folder use the pipe operator, `|`, to connect steps.

```text
input -> prompt template -> model -> output parser -> final result
```

The pipe operator means: take the output from the left side and pass it as input to the right side.

For example:

```python
chain = prompt_template | llm | StrOutputParser()
```

This means:

1. Fill the prompt template.
2. Send the completed prompt to the model.
3. Convert the model response into a plain string.


## Example 1 - Basic Chain

File: `1_chains_basics.py`

This file shows the simplest chain in the folder. It connects a prompt template, a model, and an output parser.


In [ ]:
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that can answer questions about {club_name}."),
    ("user", "What year was {club_name} founded?"),
])

chain = prompt_template | llm | StrOutputParser()

res = chain.invoke({"club_name": "Arsenal"})
print(res)

### What The Important Lines Do

- `ChatPromptTemplate.from_messages(...)` creates a reusable chat prompt.
- `{club_name}` is a placeholder that gets filled when the chain runs.
- `prompt_template | llm | StrOutputParser()` creates a pipeline.
- `chain.invoke({"club_name": "Arsenal"})` starts the chain with input data.
- `StrOutputParser()` converts the model response object into a plain string.

Key idea: a chain lets you combine multiple steps into one reusable object.


## Example 2 - Chain Inner Workings

File: `2_chain_inner_workings.py`

This file manually builds the same kind of chain using `RunnableLambda` and `RunnableSequence`. It helps beginners understand what the pipe operator is doing behind the scenes.


In [ ]:
formal_prompt = RunnableLambda(lambda x: prompt_template.format_prompt(**x))
invoke_model = RunnableLambda(lambda x: llm.invoke(x.to_messages()))
parse_output = RunnableLambda(lambda x: x.content)

manual_chain = RunnableSequence(
    first=formal_prompt,
    middle=[invoke_model],
    last=parse_output,
)

res = manual_chain.invoke({"club_name": "Arsenal"})
print(res)

### Understanding The Manual Steps

- `RunnableLambda(...)` turns a Python function into a LangChain runnable step.
- `prompt_template.format_prompt(**x)` fills the template using the input dictionary.
- `x.to_messages()` converts the prompt value into chat messages for the model.
- `llm.invoke(...)` calls the model.
- `x.content` extracts the model's answer text.
- `RunnableSequence(...)` connects those manual steps in order.

Key idea: `prompt_template | llm | StrOutputParser()` is a shorter way of building this same flow.


## Example 3 - Sequential Chaining

File: `3_sequential_chainings.py`

A sequential chain uses the output from one chain as the input to another chain. In this example, the first chain answers a question about Arsenal. The second chain translates that answer into Spanish.


In [ ]:
translation_template = ChatPromptTemplate.from_messages([
    ("system", "You are a translator and convert the provided text into {language}."),
    ("human", "Translate the following text to {language}: {text}"),
])

prepare_for_translation = RunnableLambda(
    lambda x: {"text": x, "language": "spanish"}
)

sequential_chain = (
    prompt_template
    | llm
    | StrOutputParser()
    | prepare_for_translation
    | translation_template
    | llm
    | StrOutputParser()
)

res = sequential_chain.invoke({"club_name": "Arsenal"})
print(res)

### Why `prepare_for_translation` Is Needed

After the first model call, the output is just a string. The translation prompt expects a dictionary with two keys:

```python
{"text": "...", "language": "spanish"}
```

`prepare_for_translation` reshapes the plain string into the dictionary that the next prompt needs.

Key idea: many chain bugs happen because one step outputs data in a shape the next step does not expect.


## Example 4 - Parallel Chaining

File: `4_parallel_chainings.py`

A parallel chain sends the same input into multiple branches. In this example, the movie summary is sent into one branch that analyzes the plot and another branch that analyzes the characters.


In [ ]:
summary_template = ChatPromptTemplate.from_messages([
    ("system", "You are a movie critic."),
    ("human", "Provide a brief summary of the movie {movie_name}."),
])

def analyze_plot(plot):
    plot_template = ChatPromptTemplate.from_messages([
        ("system", "You are a movie critic."),
        ("human", "Analyze the plot of the movie {plot}."),
    ])
    return plot_template.format_prompt(plot=plot)

def analyze_characters(characters):
    characters_template = ChatPromptTemplate.from_messages([
        ("system", "You are a movie critic."),
        ("human", "Analyze the characters of the movie {characters}."),
    ])
    return characters_template.format_prompt(characters=characters)


In [ ]:
def combine_verdicts(plot_analysis: str, characters_analysis: str) -> str:
    return f"Plot Analysis:\n{plot_analysis}\n\nCharacter Analysis:\n{characters_analysis}"

plot_branch_chain = RunnableLambda(lambda x: analyze_plot(x)) | llm | StrOutputParser()
characters_branch_chain = RunnableLambda(lambda x: analyze_characters(x)) | llm | StrOutputParser()

parallel_chain = (
    summary_template
    | llm
    | StrOutputParser()
    | RunnableParallel(branches={
        "plot": plot_branch_chain,
        "characters": characters_branch_chain,
    })
    | RunnableLambda(lambda x: combine_verdicts(
        x["branches"]["plot"],
        x["branches"]["characters"],
    ))
)

res = parallel_chain.invoke({"movie_name": "The Dark Knight"})
print(res)

### Understanding `RunnableParallel`

- `RunnableParallel(...)` runs multiple branches using the same input.
- The `plot` branch analyzes the movie plot.
- The `characters` branch analyzes the characters.
- The output is a dictionary containing both branch results.
- Because this file uses `RunnableParallel(branches={...})`, the results are inside `x["branches"]`.

A common mistake is trying to read `x["plot"]` when the actual shape is `x["branches"]["plot"]`.

Key idea: always know the shape of the data being passed from one chain step to the next.


## Example 5 - Conditional Chaining

File: `5_conditional_chainings.py`

A conditional chain chooses a branch based on the input. This example classifies a grade as pass or fail, then chooses a response template based on that classification.


In [ ]:
positive_response_passing_grade_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful high school teacher."),
    ("human", "Generate a positive response from the parent based on their child's grades. {grades}."),
])

negative_response_passing_grade_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful high school teacher."),
    ("human", "Generate a negative response from the parent based on their child's grades. {grades}."),
])

classification_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful high school teacher."),
    ("human", "Classify the following response into a pass or fail. If the grades {grades} are greater than or equal to 70, return 'pass'. If the grades {grades} are less than 70, return 'fail'."),
])

In [ ]:
branches = RunnableBranch(
    (
        lambda x: "pass" in x.lower(),
        positive_response_passing_grade_template | llm | StrOutputParser(),
    ),
    (
        lambda x: "fail" in x.lower(),
        negative_response_passing_grade_template | llm | StrOutputParser(),
    ),
    positive_response_passing_grade_template | llm | StrOutputParser(),
)

classification_chain = classification_template | llm | StrOutputParser()
conditional_chain = classification_chain | branches

# This demonstrates the idea, but see the note below about input shape.
# res = conditional_chain.invoke({"grades": "45"})
# print(res)

### Important Note About Input Shape

`RunnableBranch` needs a default branch at the end. That is why the final line inside `RunnableBranch(...)` is another runnable, not a condition tuple.

There is also an important data-shape issue in this example:

```text
{"grades": "45"} -> classification_chain -> "fail" -> branches
```

After `classification_chain`, the branch receives only the string `"pass"` or `"fail"`. But the positive and negative response templates still expect `{grades}`.

A cleaner design is to keep both values:

```python
{"grades": "45", "classification": "fail"}
```

Then the branch can check `classification`, while the response templates can still use `grades`.

Key idea: when chaining steps together, make sure each step receives the input format it expects.


## Folder Recap

By the end of this folder, beginners should understand:

1. A chain connects multiple LangChain steps.
2. The `|` operator passes output from one step into the next.
3. `StrOutputParser()` turns model responses into plain strings.
4. `RunnableLambda` wraps regular Python functions for use in chains.
5. `RunnableSequence` shows the manual version of a chain.
6. Sequential chains use one result as the next input.
7. Parallel chains run multiple branches at the same time.
8. Conditional chains choose a branch based on logic.
9. Many chain errors come from passing the wrong input shape to the next step.

The most important classes and functions are `ChatPromptTemplate`, `ChatOpenAI`, `StrOutputParser`, `RunnableLambda`, `RunnableSequence`, `RunnableParallel`, `RunnableBranch`, and `.invoke(...)`.
